In [1]:
import torch
import numpy as np

from dinosaw.helpers import add_custom_font, get_model, get_models, ModelTypes, model_names, get_features
from dinosaw.linear_probe import do_linear_probe, RampTypes, LinearProbeResult, get_ramp, gen_sample_mask
from dinosaw.models.vit_wrapper import PretrainedViTWrapper, MODEL_LIST, AlibiVitWrapper
import dinosaw.utils as utils
from skimage.transform import resize

from os import listdir
from PIL import Image

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from typing import Literal, TypeAlias

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'
half=False

/home/ab_aimd_anja_20884/anaconda3/envs/test-multi-gpu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
jitters=[0, 7e-6, 1e-4, 0.1]
imgs = ("black_square_1036.png", "grid.png", "random_noise_1036.png")
titles = ("black square", "grid", "random noise")

FS = 40
SF = (1, 1, 1)


In [3]:
models={
    f"{jit}":AlibiVitWrapper(model_identifier=MODEL_LIST[1], add_flash_attn=False, jitter_mag=jit).to(DEVICE) for jit in jitters}


for model_key in models.keys():
    models[model_key].load_state_dict(torch.load("../../trained_models/alibi_coco_dv2_vits14_reg_ms.pth", map_location=DEVICE))
    if half:
        models[model_key].half()

/tmp/ipykernel_3636499/1037974858.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  models[model_key].load_state_dict(torch.load("../../trained_models/alibi_coco_dv2_vits1

In [4]:
channel_group = 0
features, features_reduced = {}, {img: {} for img in imgs}
for img_file, sf_ in zip(imgs, SF):
    for model_key in models.keys():
        model = models[model_key]
        img_path = f'../images/{img_file}'
        img = Image.open(img_path).convert('RGB')
        img = img.resize((int(sf_ * img.width), int(sf_ * img.height)), Image.LANCZOS)
        feats = get_features(model, img, device=DEVICE, channel_last=False, to_half=half)#.cpu()
        # features.append(feats)
        features_reduced[img_file][model_key] = (utils.do_2D_pca(feats, (channel_group+1)*3, pre_norm="std", post_norm='minmax')[:, :, channel_group*3:channel_group*3+3])

In [5]:
%%capture
H,W=12.23,20
FLIP = False
fig, axs = plt.subplots(nrows=len(imgs), ncols=1 + len(models.keys()), figsize=(W, H))
axs = axs.ravel()
add_custom_font("resources/fonts")

for j, (img, title) in enumerate(zip(imgs, titles)):
    axs[j*(len(models.keys())+1)].imshow(Image.open(f'../images/{img}').convert("RGB"))
    axs[j*(len(models.keys())+1)].set_yticks([])
    axs[j*(len(models.keys())+1)].set_xticks([])
    if j == 1:
        axs[j*(len(models.keys())+1)].set_ylabel("ALiBi-Dv2", fontweight="bold", fontsize=FS)
    # axs[j*(len(models.keys())+1)].set_title(title, fontsize=FS)
    for i, model_key in enumerate(models.keys()):
        feats_red = features_reduced[img][model_key]
        ax = axs[j*(len(models.keys())+1) + i + 1]
        if j==0:
            ax.set_title(r"$\sigma_\text{{{jitter}}}$={value}".format(jitter="jitter", value=f"{float(model_key):.0e}" if float(model_key) <= 1e-4 and float(model_key) != 0 else f"{float(model_key)}"), fontsize=FS, fontweight = "bold" if "alibi" in model_key else None) #ALiBi-Dv2-
        ax.imshow(feats_red) #ax.imshow(np.flip(feats_red, axis=2)) if FLIP else 
        ax.set_axis_off()
plt.tight_layout()
plt.savefig('saved/S04_zero_tensor_ALiBi.jpeg', dpi=300, bbox_inches='tight', pil_kwargs={'optimize': True})

## DINOv2

In [6]:
dv2_models={f"{jit}":PretrainedViTWrapper(model_identifier=MODEL_LIST[1], add_flash_attn=False).to(DEVICE) for jit in jitters}

for model_key in dv2_models.keys():
    dv2_models[model_key].model.pos_embed.data += torch.rand_like(dv2_models[model_key].model.pos_embed.data) * float(model_key)
    if half:
        dv2_models[model_key].half()

In [7]:
channel_group = 0
features, features_reduced = {}, {img: {} for img in imgs}
for img_file, sf_ in zip(imgs, SF):
    for model_key in dv2_models.keys():
        model = dv2_models[model_key]
        img_path = f'../images/{img_file}'
        img = Image.open(img_path).convert('RGB')
        img = img.resize((int(sf_ * img.width), int(sf_ * img.height)), Image.LANCZOS)
        feats = get_features(model, img, device=DEVICE, channel_last=False, to_half=half)#.cpu()
        # features.append(feats)
        features_reduced[img_file][model_key] = (utils.do_2D_pca(feats, (channel_group+1)*3, pre_norm="std", post_norm='minmax')[:, :, channel_group*3:channel_group*3+3])

In [8]:
%%capture
H,W=12.23,20
FLIP = False
fig, axs = plt.subplots(nrows=len(imgs), ncols=1 + len(models.keys()), figsize=(W, H))
axs = axs.ravel()
add_custom_font("resources/fonts")

for j, (img, title) in enumerate(zip(imgs, titles)):
    axs[j*(len(models.keys())+1)].imshow(Image.open(f'../images/{img}').convert("RGB"))
    axs[j*(len(models.keys())+1)].set_yticks([])
    axs[j*(len(models.keys())+1)].set_xticks([])
    if j == 1:
        axs[j*(len(models.keys())+1)].set_ylabel("DINOv2", fontsize=FS)
    # axs[j*(len(models.keys())+1)].set_title(title, fontsize=FS)
    for i, model_key in enumerate(dv2_models.keys()):
        feats_red = features_reduced[img][model_key]
        ax = axs[j*(len(models.keys())+1) + i + 1]
        # ax.set_title(f"jitter={model_key}", fontsize=FS, fontweight = "bold" if "alibi" in model_key else None) #DINOv2-
        if j==0:
            ax.set_title(r"$\sigma_\text{{{jitter}}}$={value}".format(jitter="jitter", value=f"{float(model_key):.0e}" if float(model_key) <= 1e-4 and float(model_key) != 0 else f"{float(model_key)}"), fontsize=FS, fontweight = "bold" if "alibi" in model_key else None) #ALiBi-Dv2-
        ax.imshow(np.flip(feats_red, axis=2)) if FLIP else ax.imshow(feats_red)
        ax.set_axis_off()
plt.tight_layout()
plt.savefig('saved/S04_zero_tensor_Dv2.jpeg', dpi=300, bbox_inches='tight', pil_kwargs={'optimize': True})

In [9]:
%%capture
W, H = 3, 3
FS = 30
add_custom_font('resources/fonts', 'Grotesk')

N_ROWS, N_COLS = 4, 7

fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(W * N_COLS, H * N_COLS))

fig_a = Image.open('saved/S04_zero_tensor_ALiBi.jpeg')
fig_b = Image.open('saved/S04_zero_tensor_Dv2.jpeg')

axs[0].imshow(fig_a)
axs[0].axis('off')
axs[1].imshow(fig_b)
axs[1].axis('off')


labels = ['(a)', '(b)']
y_off = [1.01, 1.02]
# print(gs.subplots()[0, 0])
for i in range(2):
    ax = axs[i]
    ax.text(-0.075, y_off[i], labels[i], transform=ax.transAxes,
            fontsize=FS+6, fontweight='bold', color='black')

plt.tight_layout()
plt.savefig("saved/S04_combined.jpeg", dpi=300, bbox_inches='tight', pil_kwargs={'optimize': True})